# Text Classification with ClassifierDL - part2

**Relevant blogpost:** https://towardsdatascience.com/text-classification-in-spark-nlp-with-bert-and-universal-sentence-encoders-e644d618ca32

In [1]:
import sparknlp

spark = sparknlp.start(gpu = False) #True) # for GPU training >> sparknlp.start(gpu = True) # for Spark 2.3 =>> sparknlp.start(spark23 = True)

from sparknlp.base import *
from sparknlp.annotator import *
from pyspark.ml import Pipeline
import pandas as pd


print("Spark NLP version", sparknlp.version())
print("Apache Spark version:", spark.version)

spark

Spark NLP version 3.3.2
Apache Spark version: 3.1.2


### Load Dataset

In [2]:
trainDataset = spark.read \
      .option("header", True) \
      .csv("data/news_category_train.csv")

trainDataset.show(truncate=50)

+--------+--------------------------------------------------+
|category|                                       description|
+--------+--------------------------------------------------+
|Business| Short sellers, Wall Street's dwindling band of...|
|Business| Private investment firm Carlyle Group, which h...|
|Business| Soaring crude prices plus worries about the ec...|
|Business| Authorities have halted oil export flows from ...|
|Business| Tearaway world oil prices, toppling records an...|
|Business| Stocks ended slightly higher on Friday but sta...|
|Business| Assets of the nation's retail money market mut...|
|Business| Retail sales bounced back a bit in July, and n...|
|Business|" After earning a PH.D. in Sociology, Danny Baz...|
|Business| Short sellers, Wall Street's dwindling  band o...|
|Business| Soaring crude prices plus worries  about the e...|
|Business| OPEC can do nothing to douse scorching  oil pr...|
|Business| Non OPEC oil exporters should consider  increa...|
|Busines

In [3]:
trainDataset.count()

120000

In [4]:
from pyspark.sql.functions import col

trainDataset.groupBy("category") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

+--------+-----+
|category|count|
+--------+-----+
|Sci/Tech|30000|
|  Sports|30000|
|   World|30000|
|Business|30000|
+--------+-----+



In [5]:
from pyspark.sql.functions import col

testDataset = spark.read \
      .option("header", True) \
      .csv("data/news_category_test.csv")


testDataset.groupBy("category") \
      .count() \
      .orderBy(col("count").desc()) \
      .show()

+--------+-----+
|category|count|
+--------+-----+
|  Sports| 1900|
|Sci/Tech| 1900|
|   World| 1900|
|Business| 1900|
+--------+-----+



## ClassifierDL with Universal Sentence Embeddings

#### `sent_bert_use_cmlm_en_base` Model

In [6]:
document = DocumentAssembler()\
      .setInputCol("description")\
      .setOutputCol("document")
    
bert_cmlm = BertSentenceEmbeddings.pretrained('sent_bert_use_cmlm_en_base', 'en')\
      .setInputCols(["document"])\
      .setOutputCol("sentence_embeddings")

classsifierdl = ClassifierDLApproach()\
      .setInputCols(["sentence_embeddings"])\
      .setOutputCol("class")\
      .setLabelColumn("category")\
      .setMaxEpochs(10)\
      .setEnableOutputLogs(True)\
      .setLr(0.001)

bert_cmlm_clf_pipeline = Pipeline(
    stages = [
        document,
        bert_cmlm,
        classsifierdl
    ])

sent_bert_use_cmlm_en_base download started this may take some time.
Approximate size to download 391.6 MB
[OK!]


In [ ]:
#%%time
bert_cmlm_pipelineModel = bert_cmlm_clf_pipeline.fit(trainDataset)

In [ ]:
from sklearn.metrics import classification_report

preds = bert_cmlm_pipelineModel.transform(testDataset)

preds_df = preds.select('category','description',"class.result").toPandas()

preds_df['result'] = preds_df['result'].apply(lambda x : x[0])

print (classification_report(preds_df['result'], preds_df['category']))


              precision    recall  f1-score   support

    Business       0.85      0.88      0.87      1838
    Sci/Tech       0.91      0.85      0.88      2021
      Sports       0.98      0.95      0.97      1949
       World       0.88      0.94      0.91      1792

    accuracy                           0.91      7600
   macro avg       0.91      0.91      0.91      7600
weighted avg       0.91      0.91      0.91      7600



In [ ]:
log_files = os.listdir("C:/Users/chang/annotator_logs")

with open("C:/Users/chang/annotator_logs/"+log_files[0], "r") as log_file :
    print(log_file.read())

Training started - epochs: 10 - learning_rate: 0.001 - batch_size: 64 - training_examples: 120000 - classes: 4
Epoch 0/10 - 7.95s - loss: 1628.9716 - acc: 0.8763 - batches: 1875
Epoch 1/10 - 7.25s - loss: 1603.904 - acc: 0.887075 - batches: 1875
Epoch 2/10 - 7.13s - loss: 1598.0989 - acc: 0.8906583 - batches: 1875
Epoch 3/10 - 7.10s - loss: 1594.3523 - acc: 0.893325 - batches: 1875
Epoch 4/10 - 7.07s - loss: 1591.4575 - acc: 0.895725 - batches: 1875
Epoch 5/10 - 7.12s - loss: 1588.7573 - acc: 0.89725 - batches: 1875
Epoch 6/10 - 7.11s - loss: 1586.2809 - acc: 0.89880836 - batches: 1875
Epoch 7/10 - 7.06s - loss: 1584.1012 - acc: 0.9000583 - batches: 1875
Epoch 8/10 - 7.07s - loss: 1582.1438 - acc: 0.90125 - batches: 1875
Epoch 9/10 - 7.05s - loss: 1580.3627 - acc: 0.9023333 - batches: 1875



## Getting prediction from Trained model

In [ ]:
from sparknlp.base import LightPipeline

light_model = LightPipeline(use_pipelineModel)

In [ ]:
testDataset.select('description').take(2)

[Row(description="Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul."),
 Row(description=' TORONTO, Canada    A second team of rocketeers competing for the  #36;10 million Ansari X Prize, a contest for privately funded suborbital space flight, has officially announced the first launch date for its manned rocket.')]

In [ ]:
text='''
Fearing the fate of Italy, the centre-right government has threatened to be merciless with those who flout tough restrictions. 
As of Wednesday it will also include all shops being closed across Greece, with the exception of supermarkets. Banks, pharmacies, pet-stores, mobile phone stores, opticians, bakers, mini-markets, couriers and food delivery outlets are among the few that will also be allowed to remain open.
'''
result = light_model.annotate(text)

result['class']

['World']

In [ ]:
light_model.annotate('the soccer games will be postponed.')

{'class': ['Sports'],
 'document': ['the soccer games will be postponed.'],
 'sentence_embeddings': ['the soccer games will be postponed.']}